# PCL detection: class weights only

RoBERTa-base with HTML stripping (as in baseline). **Imbalance handling:** class-weighted cross-entropy loss only (no sampling, no focal loss).

In [1]:
import html
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
USE_CLASS_WEIGHTS = True  # class-weighted CE loss only

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Resolve project root regardless of whether notebook is run from project root or src/
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing data/ directory.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_DIR = PROJECT_ROOT / "models" / "roberta_pcl_imbalance_weights_only"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_PREFIX = "roberta_pcl_imbalance_weights_only"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

/vol/bitbucket/tq422/nlpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Project root: /vol/bitbucket/tq422/NLP_cw


## Load data and build train/dev splits

In [2]:
df = pd.read_csv(
    RAW_DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
)

df["label_binary"] = df["label"].apply(lambda x: 0 if x in [0, 1] else 1)

# Split IDs (match organizer: exactly one row per ID from split files)
a_train = pd.read_csv(RAW_DATA_DIR / "train_semeval_parids-labels.csv")
a_dev = pd.read_csv(RAW_DATA_DIR / "dev_semeval_parids-labels.csv")

df_sub = df[["par_id", "text", "label_binary"]].drop_duplicates(subset=["par_id"], keep="first").copy()
df_sub["text"] = df_sub["text"].fillna("").astype(str)

train_df = a_train[["par_id"]].merge(df_sub, on="par_id", how="left")
dev_df = a_dev[["par_id"]].merge(df_sub, on="par_id", how="left")
train_df["text"] = train_df["text"].fillna("").astype(str)
dev_df["text"] = dev_df["text"].fillna("").astype(str)
train_df["label_binary"] = train_df["label_binary"].fillna(0).astype(int)
dev_df["label_binary"] = dev_df["label_binary"].fillna(0).astype(int)

# Split a validation set from training data; dev is held out and not used during training
VAL_RATIO = 0.1
train_df, val_df = train_test_split(
    train_df, test_size=VAL_RATIO, stratify=train_df["label_binary"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Dev size:   {len(dev_df)} (held out, not used during training)")

# Load test set (no labels); preprocessed in next section
test_df = pd.read_csv(
    RAW_DATA_DIR / "task4_test.tsv",
    sep="\t",
    names=["id", "par_id", "keyword", "country_code", "text"],
)
test_df["text"] = test_df["text"].fillna("").astype(str)
print(f"Test size:  {len(test_df)}")

print("\nTrain label distribution:")
print(train_df["label_binary"].value_counts())
print("\nVal label distribution:")
print(val_df["label_binary"].value_counts())
print("\nDev label distribution:")
print(dev_df["label_binary"].value_counts())

Train size: 7537
Val size:   838
Dev size:   2094 (held out, not used during training)

Train label distribution:
label_binary
0    6822
1     715
Name: count, dtype: int64

Val label distribution:
label_binary
0    759
1     79
Name: count, dtype: int64

Dev label distribution:
label_binary
0    1895
1     199
Name: count, dtype: int64


## Preprocess text: remove HTML markups

From data exploration: texts can contain HTML tags (e.g. `<h>`, `</p>`) that do not add useful signal. Strip tags and unescape HTML entities, then normalize whitespace.

In [3]:
def strip_html_and_clean(text: str) -> str:
    """Remove HTML tags, unescape entities, and normalize whitespace."""
    if not isinstance(text, str) or not text.strip():
        return text.strip() if isinstance(text, str) else ""
    # Remove HTML tags (pattern from data_exploration: <[^>]+>)
    no_tags = re.sub(r"<[^>]+>", "", text)
    # Unescape HTML entities (e.g. &amp; -> &, &lt; -> <)
    unescaped = html.unescape(no_tags)
    # Collapse multiple spaces and strip
    return re.sub(r"\s+", " ", unescaped).strip()


train_df["text"] = train_df["text"].apply(strip_html_and_clean)
val_df["text"] = val_df["text"].apply(strip_html_and_clean)
dev_df["text"] = dev_df["text"].apply(strip_html_and_clean)
test_df["text"] = test_df["text"].apply(strip_html_and_clean)

# Sanity check: ensure no empty texts
train_df["text"] = train_df["text"].replace("", " ")
val_df["text"] = val_df["text"].replace("", " ")
dev_df["text"] = dev_df["text"].replace("", " ")
test_df["text"] = test_df["text"].replace("", " ")
print("HTML preprocessing applied to train, val, dev, and test texts.")

HTML preprocessing applied to train, val, and dev texts.


In [4]:
# Class weights for loss only (inverse frequency; no sampling)
n0 = (train_df["label_binary"] == 0).sum()
n1 = (train_df["label_binary"] == 1).sum()
n = n0 + n1
w0 = n / (2 * n0)
w1 = n / (2 * n1)
class_weights = torch.tensor([w0, w1], dtype=torch.float32)

print(f"Class weights (0, 1): {class_weights.tolist()}")

Class weights (0, 1): [0.55240398645401, 5.270629405975342]


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = PCLDataset(train_df["text"], train_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
val_dataset = PCLDataset(val_df["text"], val_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
dev_dataset = PCLDataset(dev_df["text"], dev_df["label_binary"], tokenizer, max_length=MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer and datasets ready.")

Tokenizer and datasets ready.


## Training

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average=None, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = (preds == labels).mean()

    return {
        "accuracy": float(acc),
        "precision_pos": float(p_pos[0]),
        "recall_pos": float(r_pos[0]),
        "f1_pos": float(f1_pos[0]),
        "f1_macro": float(f1_macro),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)


class ImbalanceTrainer(Trainer):
    """Trainer with class-weighted CE loss only (no sampling)."""

    def __init__(self, class_weights=None, use_class_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
        self.use_class_weights = use_class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.use_class_weights and self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


trainer = ImbalanceTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    class_weights=class_weights if USE_CLASS_WEIGHTS else None,
    use_class_weights=USE_CLASS_WEIGHTS,
)

print("Trainer initialized.")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainer initialized.


In [7]:
train_result = trainer.train()
print("Training finished.")

Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.506900,0.635919,0.563246,0.173121,0.962025,0.293436,0.488687
2,0.421500,0.476204,0.884248,0.428571,0.683544,0.526829,0.730444
3,0.360100,0.979816,0.912888,0.545455,0.455696,0.496552,0.724435
4,0.211800,0.957356,0.911695,0.526316,0.632911,0.574713,0.762723
5,0.096500,1.782654,0.909308,0.526316,0.379747,0.441176,0.695913
6,0.207700,1.708096,0.911695,0.535211,0.481013,0.506667,0.729087
7,0.013400,2.162454,0.906921,0.507246,0.443038,0.472973,0.710963


Training finished.


In [8]:
eval_metrics = trainer.evaluate(dev_dataset)
print("Dev metrics:")
for k, v in eval_metrics.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

pred_output = trainer.predict(dev_dataset)
dev_preds = np.argmax(pred_output.predictions, axis=-1)
dev_labels = pred_output.label_ids

import json
report_str = classification_report(dev_labels, dev_preds, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (dev):")
print(report_str)

dev_metrics_record = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in eval_metrics.items()}
dev_metrics_record["classification_report"] = report_str
dev_metrics_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_dev_metrics.json"
with open(dev_metrics_path, "w", encoding="utf-8") as f:
    json.dump(dev_metrics_record, f, indent=2)
print(f"Saved dev metrics to: {dev_metrics_path}")

dev_txt_path = OUTPUT_DIR / "dev.txt"
with open(dev_txt_path, "w", encoding="utf-8") as f:
    f.write("label_binary\tpred_label\n")
    for lb, pr in zip(dev_labels, dev_preds):
        f.write(f"{lb}\t{pr}\n")
print(f"Saved dev predictions to: {dev_txt_path}")
with open(PROJECT_ROOT / "dev.txt", "w", encoding="utf-8") as f:
    for pr in dev_preds:
        f.write(f"{pr}\n")
print(f"Saved dev predictions to {PROJECT_ROOT / 'dev.txt'} (root)")

Dev metrics:
eval_loss: 0.3124
eval_accuracy: 0.9236
eval_precision_pos: 0.5924
eval_recall_pos: 0.6281
eval_f1_pos: 0.6098
eval_f1_macro: 0.7837
eval_runtime: 6.9695
eval_samples_per_second: 300.4530
eval_steps_per_second: 9.4700
epoch: 7.0000

Classification report (dev):
              precision    recall  f1-score   support

      No PCL     0.9607    0.9546    0.9576      1895
         PCL     0.5924    0.6281    0.6098       199

    accuracy                         0.9236      2094
   macro avg     0.7766    0.7914    0.7837      2094
weighted avg     0.9257    0.9236    0.9246      2094

Saved dev metrics to: /vol/bitbucket/tq422/NLP_cw/output/roberta_pcl_imbalance_weights_only_dev_metrics.json
Saved dev predictions to: /vol/bitbucket/tq422/NLP_cw/output/dev.txt
Saved dev predictions to /vol/bitbucket/tq422/NLP_cw/dev.txt (root)


In [9]:
# Save best model and tokenizer
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

print(f"Saved model and tokenizer to: {MODEL_DIR}")

Saved model and tokenizer to: /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_imbalance_weights_only


In [ ]:
# Optional: produce predictions for official test split (labels are hidden)
# test_df already loaded and preprocessed in data-loading and preprocessing cells
test_dataset = PCLDataset(
    texts=test_df["text"],
    labels=np.zeros(len(test_df), dtype=int),  # placeholder labels for dataset interface
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_pred_output = trainer.predict(test_dataset)
test_preds = np.argmax(test_pred_output.predictions, axis=-1)

test_txt_path = OUTPUT_DIR / "test.txt"
with open(test_txt_path, "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions to: {test_txt_path}")
with open(PROJECT_ROOT / "test.txt", "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions to {PROJECT_ROOT / 'test.txt'} (root)")

Saved test predictions to: /vol/bitbucket/tq422/NLP_cw/output/test.txt
Saved test predictions to /vol/bitbucket/tq422/NLP_cw/test.txt (root)


: 